In [25]:
import cv2
import time
import numpy as np
import pandas as pd

from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

import motmetrics as mm

In [37]:


VIDEO_SOURCE   = r"D:/obj detection/train/MOT17-02-FRCNN/img1"

MODEL_PATH     = "yolov8l.pt"

TRACKER_TYPE   = "deepsort"     # "bytesort" or "deepsort"

SAVE_RESULT_PATH = f"{TRACKER_TYPE}_results.txt"

TARGET_CLASS   = 0             





In [38]:
model = YOLO(MODEL_PATH)

if TRACKER_TYPE == "deepsort":
    tracker = DeepSort(
        max_age=50,
        
        n_init=1,
    
        max_cosine_distance=0.3,
        embedder="mobilenet",
        embedder_gpu=True
    )

In [39]:
import os

image_files = sorted([
    os.path.join(VIDEO_SOURCE, img)
    for img in os.listdir(VIDEO_SOURCE)
    if img.endswith(".jpg")
])

print("Total Frames:", len(image_files))

Total Frames: 600


In [40]:
tracking_results = []
frame_id = 0
total_fps = []

for image_path in image_files:

    frame = cv2.imread(image_path)

    if frame is None:
        continue

    frame_id += 1
    start_time = time.time()

   
    # BYTE TRACK
  
    if TRACKER_TYPE == "bytesort":
        results = model.track(
            frame,
            persist=True,
            imgsz=1280,
            conf=0.10,
            tracker="bytetrack.yaml",
            device="cuda",
            verbose=False
        )[0]

        if results.boxes.id is not None:

            boxes      = results.boxes.xyxy.int().cpu().tolist()
            track_ids  = results.boxes.id.int().cpu().tolist()
            classes    = results.boxes.cls.int().cpu().tolist()
            confs      = results.boxes.conf.cpu().tolist()

            print(
                f"Frame {frame_id}: "
                f"Detections={len(boxes)} "
                f"Tracks={len(track_ids)}"
            )

            for box, track_id, cls, conf in zip(boxes, track_ids, classes, confs):

                if cls != TARGET_CLASS:
                    continue

                x1, y1, x2, y2 = box
                w = x2 - x1
                h = y2 - y1

                tracking_results.append([
                    frame_id, track_id,
                    x1, y1, w, h,
                    round(conf, 4),
                    -1, -1, -1
                ])

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    frame,
                    f"{model.names[cls]} ID:{track_id}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2
                )

  
    # DEEP SORT
   
    elif TRACKER_TYPE == "deepsort":

        results = model(
            frame,
            imgsz=1280,
            conf=0.10,  
            device="cuda",
            verbose=False
        )[0]

        detections = []

        for box in results.boxes:

            class_id   = int(box.cls[0].cpu().item())
            confidence = float(box.conf[0].cpu().item())

            if class_id != TARGET_CLASS:
                continue

            
         
            x1, y1, x2, y2 = box.xyxy[0].int().cpu().tolist()

            detections.append((
                [x1, y1, x2 - x1, y2 - y1],
                confidence,
                class_id
            ))

        tracks = tracker.update_tracks(detections, frame=frame)

        print(
            f"Frame {frame_id}: "
            f"Detections={len(detections)} "
            f"Tracks={len(tracks)}"
        )

        for track in tracks:

           
            if not track.is_confirmed():
                continue

           
            if track.time_since_update > 1:
                continue

            x1, y1, x2, y2 = map(int, track.to_ltrb())
            w = x2 - x1
            h = y2 - y1

            cls = getattr(track, "det_class",
                  getattr(track, "class_id", None))

          
            track_conf = track.det_conf if track.det_conf is not None else 1.0

            tracking_results.append([
                frame_id,
                track.track_id,
                x1, y1, w, h,
                round(float(track_conf), 4),
                -1, -1, -1
            ])

            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(
                frame,
                f"{model.names[cls] if cls is not None else 'Person'} ID:{track.track_id}",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2
            )

    
    # FPS & DISPLAY

    fps = 1 / (time.time() - start_time)
    total_fps.append(fps)

    cv2.putText(
        frame,
        f"FPS: {int(fps)}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2
    )
    cv2.putText(
    frame,
    f"Frame: {frame_id}",
    (20, 80),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.8,
    (255, 255, 0),
    2
   )

    cv2.imshow("Tracking", frame)

    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()
print("Tracking Completed")

Frame 1: Detections=26 Tracks=26
Frame 2: Detections=24 Tracks=24
Frame 3: Detections=25 Tracks=26
Frame 4: Detections=30 Tracks=31
Frame 5: Detections=28 Tracks=29
Frame 6: Detections=28 Tracks=31
Frame 7: Detections=27 Tracks=30
Frame 8: Detections=29 Tracks=31
Frame 9: Detections=31 Tracks=33
Frame 10: Detections=28 Tracks=33
Frame 11: Detections=28 Tracks=33
Frame 12: Detections=32 Tracks=37
Frame 13: Detections=28 Tracks=36
Frame 14: Detections=28 Tracks=34
Frame 15: Detections=32 Tracks=40
Frame 16: Detections=30 Tracks=39
Frame 17: Detections=32 Tracks=43
Frame 18: Detections=25 Tracks=39
Frame 19: Detections=26 Tracks=40
Frame 20: Detections=29 Tracks=42
Frame 21: Detections=30 Tracks=44
Frame 22: Detections=32 Tracks=43
Frame 23: Detections=24 Tracks=42
Frame 24: Detections=26 Tracks=42
Frame 25: Detections=24 Tracks=42
Frame 26: Detections=23 Tracks=44
Frame 27: Detections=26 Tracks=45
Frame 28: Detections=28 Tracks=45
Frame 29: Detections=27 Tracks=45
Frame 30: Detections=28

In [41]:
df = pd.DataFrame(tracking_results)

df.to_csv(
    SAVE_RESULT_PATH,
    header=False,
    index=False
)
print("Rows written:", len(tracking_results))
print("Results saved to:", SAVE_RESULT_PATH)

Rows written: 16107
Results saved to: deepsort_results.txt


In [42]:
avg_fps = sum(total_fps) / len(total_fps)
print("Average FPS:", round(avg_fps, 2))

Average FPS: 2.58


In [43]:
gt_path = r"D:/obj detection/train/MOT17-02-FRCNN/gt/gt.txt"

gt = pd.read_csv(gt_path, header=None)

gt.columns = ["frame", "id", "x", "y", "w", "h", "conf", "class", "visibility"]


gt = gt[
    (gt["class"] == 1) &
    (gt["conf"]  == 1)
]

print("GT rows:", len(gt))
gt.head()

GT rows: 18581


,frame,id,x,y,w,h,conf,class,visibility
600,1,2,1338,418,167,379,1,1,1.0
601,2,2,1342,417,168,380,1,1,1.0
602,3,2,1346,417,170,380,1,1,1.0
603,4,2,1351,417,171,381,1,1,1.0
604,5,2,1355,417,173,381,1,1,1.0


In [44]:
pred = pd.read_csv(SAVE_RESULT_PATH, header=None)

pred.columns = ["frame", "id", "x", "y", "w", "h", "conf", "a", "b", "c"]

print("Pred rows:", len(pred))
pred.head()

Pred rows: 16107


,frame,id,x,y,w,h,conf,a,b,c
0,2,1,1345,419,166,373,0.8917,-1,-1,-1
1,2,2,583,444,89,263,0.8809,-1,-1,-1
2,2,3,440,443,112,280,0.8638,-1,-1,-1
3,2,4,1433,430,170,337,0.7536,-1,-1,-1
4,2,5,1015,428,43,101,0.8206,-1,-1,-1


In [45]:
print("GT objects   :", len(gt))
print("Pred objects :", len(pred))
print()
print("GT unique frames  :", gt["frame"].nunique())
print("Pred unique frames:", pred["frame"].nunique())
print()


GT objects   : 18581
Pred objects : 16107

GT unique frames  : 600
Pred unique frames: 599



In [46]:
acc = mm.MOTAccumulator(auto_id=True)


all_frames = sorted(
    set(gt["frame"].unique()) | set(pred["frame"].unique())
)

for frame in all_frames:

    gt_frame   = gt[gt["frame"]     == frame]
    pred_frame = pred[pred["frame"] == frame]

    gt_ids   = gt_frame["id"].tolist()
    pred_ids = pred_frame["id"].tolist()

    gt_boxes   = gt_frame[["x", "y", "w", "h"]].values
    pred_boxes = pred_frame[["x", "y", "w", "h"]].values

   
    distance_matrix = mm.distances.iou_matrix(
        gt_boxes,
        pred_boxes,
        max_iou=0.5
    )

    acc.update(
        gt_ids,
        pred_ids,
        distance_matrix
    )

mh = mm.metrics.create()

summary = mh.compute(
    acc,
    metrics=[
        "mota",
        "idf1",
        "num_switches",
        "precision",
        "recall",
        "num_false_positives",
        "num_misses"
    ],
    name=TRACKER_TYPE
)

print(summary)

              mota     idf1  num_switches  precision    recall  \
deepsort  0.094613  0.29428           307   0.564103  0.488994   

          num_false_positives  num_misses  
deepsort                 7021        9495  


In [13]:
summary.to_csv(f"{TRACKER_TYPE}_evaluation.csv")
print("Evaluation saved to:", f"{TRACKER_TYPE}_evaluation.csv")

Evaluation saved to: deepsort_evaluation.csv
